---
### Inisialisasi Lingkungan & SparkSession

Mengatur konsistensi interpreter driver dan worker Spark serta menginisialisasi `SparkSession` lokal dengan partisi shuffle optimal.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Memastikan interpreter Python driver dan worker selaras
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas11_2505060072_Muhammad_Raka_Mushaddaq") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil diinisialisasi. Versi Spark:", spark.version)

26/11/12 17:52:32 WARN Utils: Your hostname, raka-Aspire-A314-32 resolves to a loopback address: 127.0.1.1; using 10.90.70.61 instead (on interface wlp3s0)
26/11/12 17:52:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/12 17:52:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil diinisialisasi. Versi Spark: 3.5.9


---
### Menyiapkan & Membaca Dataset Penjualan

Membuat dataset simulasi pesanan `tugas11_penjualan.csv` (jika belum ada) dan memuatnya ke dalam Spark DataFrame menggunakan skema inferensi otomatis.


In [2]:
# Membuat dataset tugas11_penjualan.csv sesuai instruksi modul
csv_path = "tugas11_penjualan.csv"
if not os.path.exists(csv_path):
    np.random.seed(111)
    n = 2500
    menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
    cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
    waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

    data_tugas11 = pd.DataFrame({
        "nama_menu": np.random.choice(menu_list, size=n),
        "cabang": np.random.choice(cabang_list, size=n),
        "waktu_pesan": np.random.choice(waktu_list, size=n),
        "qty": np.random.randint(1, 6, size=n),
        "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
    })
    data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
    data_tugas11.to_csv(csv_path, index=False)
    print(f"Berkas {csv_path} berhasil dibuat!")

# Membaca berkas CSV ke dalam Spark DataFrame
df = spark.read.csv(csv_path, header=True, inferSchema=True)
print(f"Dataset berhasil dimuat ke DataFrame. Total baris: {df.count()}")
df.printSchema()
df.show(5)

Dataset berhasil dimuat ke DataFrame. Total baris: 2500
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



---
### Bagian A. Feature Engineering 

**Instruksi:**
> Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.


In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Definisi StringIndexer untuk tiap fitur kategori teks
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# 2. Fit StringIndexer model ke DataFrame data pelatihan
model_indexer_menu = indexer_menu.fit(df)
model_indexer_cabang = indexer_cabang.fit(df)
model_indexer_waktu = indexer_waktu.fit(df)

# 3. Menerapkan transformasi indexing
df_indexed = model_indexer_waktu.transform(
    model_indexer_cabang.transform(
        model_indexer_menu.transform(df)
    )
)

# 4. Menggabungkan 5 fitur menjadi satu kolom Vector tunggal 'features'
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler_utama.transform(df_indexed)

# Menampilkan hasil transformasi feature engineering
print("Hasil Feature Engineering:")
df_final.select("features", "total_penjualan").show(8, truncate=False)

Hasil Feature Engineering:


+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
|(5,[0,1],[1.0,30000.0])  |32462.966390290698|
|[2.0,25000.0,2.0,0.0,1.0]|51846.84285104993 |
|[5.0,25000.0,2.0,2.0,0.0]|114192.26376783242|
+-------------------------+------------------+
only showing top 8 rows



---
### Bagian B. Train/Test Split & Pelatihan Model 

**Instruksi:**
> Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.


In [4]:
from pyspark.ml.regression import LinearRegression

# 1. Membagi data menjadi 75% training dan 25% testing dengan seed=101
train_df, test_df = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah Data Training : {train_df.count()} ({train_df.count()/df_final.count()*100:.1f}%)")
print(f"Jumlah Data Testing  : {test_df.count()} ({test_df.count()/df_final.count()*100:.1f}%)")

# 2. Mendefinisikan model LinearRegression
lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")

# 3. Melatih model menggunakan data training
model_utama = lr_utama.fit(train_df)

print("\nModel Linear Regression berhasil dilatih!")
print("Koefisien tiap fitur [qty, harga_satuan, menu_idx, cabang_idx, waktu_idx]:")
print(model_utama.coefficients)
print(f"Intercept: {model_utama.intercept:,.2f}")

Jumlah Data Training : 1839 (73.6%)


Jumlah Data Testing  : 661 (26.4%)



Model Linear Regression berhasil dilatih!
Koefisien tiap fitur [qty, harga_satuan, menu_idx, cabang_idx, waktu_idx]:
[22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66,903.18


---
### Bagian C. Evaluasi Model 

**Instruksi:**
> Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?


In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Menghasilkan prediksi pada data testing
pred_utama = model_utama.transform(test_df)

# 2. Evaluasi RMSE dan R²
eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = eval_rmse.evaluate(pred_utama)
r2_utama = eval_r2.evaluate(pred_utama)

print("=== HASIL EVALUASI MODEL UTAMA (BAGIAN C) ===")
print(f"RMSE : Rp {rmse_utama:,.2f}")
print(f"R²   : {r2_utama:.4f} ({r2_utama*100:.2f}% variasi data dijelaskan oleh model)")

# 3. Menampilkan sampel perbandingan nilai aktual vs prediksi
print("\nSampel Hasil Prediksi vs Nilai Aktual:")
pred_utama.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "total_penjualan", "prediction"
).show(10)

=== HASIL EVALUASI MODEL UTAMA (BAGIAN C) ===
RMSE : Rp 10,900.80
R²   : 0.9107 (91.07% variasi data dijelaskan oleh model)

Sampel Hasil Prediksi vs Nilai Aktual:


+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31056.146105506836|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31056.146105506836|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110174.03658512786|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732|119242.79918960987|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318| 67877.77492201277|
|Americano|Magelang|       Pagi|

**Interpretasi Evaluasi:**
Model regresi ini menghasilkan nilai **R² sebesar 0.9107**, yang menunjukkan bahwa lebih dari **91.07% variasi total penjualan pesanan berhasil dijelaskan secara akurat oleh fitur-fitur di dalam model**, dengan nilai kesalahan rata-rata (**RMSE**) sebesar **Rp 10.900,80** yang sangat mendekati simpangan baku variasi acak bawaan data. Oleh karena itu, model ini **sangat layak dan cukup baik** digunakan oleh pihak manajemen "Kopi Nusantara" sebagai alat bantu estimasi omzet harian maupun perencanaan penyediaan stok bahan baku di setiap cabang.

---
### Bagian D. Eksperimen Perbandingan Fitur 

**Instruksi:**
> Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?


In [6]:
# 1. Membuat VectorAssembler kedua HANYA dengan 2 fitur: qty & harga_satuan
assembler_kedua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_kedua"
)

df_final_kedua = assembler_kedua.transform(df)

# 2. Split data menggunakan rasio dan seed yang sama persis (75% : 25%, seed=101)
train_df_2, test_df_2 = df_final_kedua.randomSplit([0.75, 0.25], seed=101)

# 3. Melatih model kedua
lr_kedua = LinearRegression(featuresCol="features_kedua", labelCol="total_penjualan")
model_kedua = lr_kedua.fit(train_df_2)

# 4. Melakukan prediksi dan evaluasi pada data test model kedua
pred_kedua = model_kedua.transform(test_df_2)
rmse_kedua = eval_rmse.evaluate(pred_kedua)
r2_kedua = eval_r2.evaluate(pred_kedua)

print("=== PERBANDINGAN MODEL UTAMA VS MODEL KEDUA ===")
print(f"Model Utama (5 fitur)     -> RMSE: Rp {rmse_utama:,.2f} | R²: {r2_utama:.4f}")
print(f"Model Kedua (2 fitur)     -> RMSE: Rp {rmse_kedua:,.2f} | R²: {r2_kedua:.4f}")
print(f"Selisih R² (Model 2 - 1)  : {r2_kedua - r2_utama:+.6f}")
print(f"Selisih RMSE (Model 2 - 1): Rp {rmse_kedua - rmse_utama:+,.2f}")

=== PERBANDINGAN MODEL UTAMA VS MODEL KEDUA ===
Model Utama (5 fitur)     -> RMSE: Rp 10,900.80 | R²: 0.9107
Model Kedua (2 fitur)     -> RMSE: Rp 10,896.03 | R²: 0.9108
Selisih R² (Model 2 - 1)  : +0.000078
Selisih RMSE (Model 2 - 1): Rp -4.77


**Kesimpulan Analisis Perbandingan Fitur:**
Berdasarkan hasil eksperimen di atas, model kedua yang hanya menggunakan 2 fitur inti (`qty` dan `harga_satuan`) menghasilkan **R² = 0.9108** dan **RMSE = Rp 10.896,03**, yang hampir identik bahkan sedikit lebih baik daripada model utama dengan 5 fitur (**R² = 0.9107** dan **RMSE = Rp 10.900,80**).
Dapat disimpulkan bahwa **penambahan informasi `nama_menu`, `cabang`, dan `waktu_pesan` TIDAK meningkatkan akurasi prediksi secara berarti**.
Alasan mendasarnya adalah bahwa total penjualan pesanan secara alamiah terbentuk langsung dari operasi matematis `qty * harga_satuan` ditambah variasi acak (*Gaussian noise*). Kategori nama menu, cabang lokasi, dan waktu pesan bersifat independen terhadap nilai penjualan (selain dari keterkaitan harga satuannya), sehingga penambahan ketiga fitur kategorikal tersebut hanya menambah kompleksitas model dan dimensi noise tanpa memberikan sinyal prediktif baru.

---
### Bagian E. Prediksi Skenario Baru 

**Instruksi:**
> Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.


In [7]:
# 1. Menyiapkan DataFrame baru berisi 3 skenario pesanan dummy
data_skenario_dummy = [
    ("Americano", "Magelang", "Pagi", 2, 18000),
    ("Kopi Susu", "Yogyakarta", "Sore", 4, 20000),
    ("Roti Bakar", "Semarang", "Malam", 3, 25000)
]
kolom_skenario = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(data_skenario_dummy, kolom_skenario)

print("Data Skenario Baru (Sebelum Transformasi):")
df_skenario.show()

# 2. Transformasi menggunakan model StringIndexer yang SAMA (yang sudah di-fit pada Bagian A)
# PENTING: Gunakan model_indexer_* yang sudah di-fit sebelumnya, JANGAN memanggil .fit() baru!
df_skenario_indexed = model_indexer_waktu.transform(
    model_indexer_cabang.transform(
        model_indexer_menu.transform(df_skenario)
    )
)

# 3. Transformasi fitur menggunakan VectorAssembler utama yang sama
df_skenario_features = assembler_utama.transform(df_skenario_indexed)

# 4. Menjalankan prediksi menggunakan model LinearRegression utama
prediksi_skenario = model_utama.transform(df_skenario_features)

print("=== HASIL PREDIKSI TOTAL PENJUALAN UNTUK 3 SKENARIO BARU ===")
prediksi_skenario.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

Data Skenario Baru (Sebelum Transformasi):


+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Americano|  Magelang|       Pagi|  2|       18000|
| Kopi Susu|Yogyakarta|       Sore|  4|       20000|
|Roti Bakar|  Semarang|      Malam|  3|       25000|
+----------+----------+-----------+---+------------+



=== HASIL PREDIKSI TOTAL PENJUALAN UNTUK 3 SKENARIO BARU ===


+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Americano |Magelang  |Pagi       |2  |18000       |31602.72450408472|
|Kopi Susu |Yogyakarta|Sore       |4  |20000       |82916.87943705732|
|Roti Bakar|Semarang  |Malam      |3  |25000       |75215.1846728456 |
+----------+----------+-----------+---+------------+-----------------+



**Pembahasan Hasil Prediksi Skenario Baru:**
Hasil prediksi model untuk ketiga skenario pesanan baru menunjukkan keselarasan yang sangat tinggi dengan kalkulasi bisnis aktual:
1. **Skenario 1 (2 Americano @ Rp 18.000):**
   - Nilai dasar teoritis: $2 \times 18.000 = \text{Rp } 36.000$.
   - Hasil prediksi model: **Rp 31.602,72**.
2. **Skenario 2 (4 Kopi Susu @ Rp 20.000):**
   - Nilai dasar teoritis: $4 \times 20.000 = \text{Rp } 80.000$.
   - Hasil prediksi model: **Rp 82.916,88**.
3. **Skenario 3 (3 Roti Bakar @ Rp 25.000):**
   - Nilai dasar teoritis: $3 \times 25.000 = \text{Rp } 75.000$.
   - Hasil prediksi model: **Rp 75.215,18**.

Seluruh estimasi model berada dalam rentang toleransi noise acak yang sangat wajar (margin kesalahan tipis), membuktikan bahwa model regresi terdistribusi yang dibangun melalui Spark MLlib berhasil menangkap pola fungsi penentuan harga pesanan dengan sangat presisi.


---
### Penutupan SparkSession

Memastikan seluruh sumber daya Spark dilepaskan secara bersih sebelum pengumpulan notebook.


In [8]:
# Menutup SparkSession
spark.stop()
print("SparkSession berhasil ditutup secara bersih. Seluruh tahapan selesai!")

SparkSession berhasil ditutup secara bersih. Seluruh tahapan selesai!
